# What is PySpark?

PySpark is a Python API for Apache Spark used to:

- Process large datasets
- Transform data
- Analyze data

In simple terms: **PySpark helps us work with big data efficiently**

---

# What is CRUD?

CRUD stands for:

| Operation | Meaning | PySpark Equivalent |
| --- | --- | --- |
| C | Create | Create DataFrame / Add rows |
| R | Read | Select / Filter |
| U | Update | withColumn |
| D | Delete | Filter (remove rows) |

---

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS b2practice.pyspark")

DataFrame[]

# 1. CREATE → Add Data

### Purpose:

Create or add data into a DataFrame

In [0]:
data = [
    (1,"John Smith","UK","2024-01-10"),
    (2,"Alice Brown","USA","2024-02-15"),
    (3,"Raj Patel","India","2024-03-01")
]

columns= ["customer_id","name","country","signup_date"]

df=spark.createDataFrame(data,columns)
df.show()

+-----------+-----------+-------+-----------+
|customer_id|       name|country|signup_date|
+-----------+-----------+-------+-----------+
|          1| John Smith|     UK| 2024-01-10|
|          2|Alice Brown|    USA| 2024-02-15|
|          3|  Raj Patel|  India| 2024-03-01|
+-----------+-----------+-------+-----------+



In [0]:
#A PySpark DataFrame is immutable, so you don't directly insert a row into df. Instead, create another DataFrame for the new row and union it with the existing one.

new_data = [
    (4, "Emma Wilson", "UK", "2024-04-01"),
    (5, "David Lee", "Canada", "2024-05-10")
]

new_df = spark.createDataFrame(new_data, columns)

df = df.union(new_df)

In [0]:
#df.display()
#df.show()
df.select("*").display()

customer_id,name,country,signup_date
1,John Smith,UK,2024-01-10
2,Alice Brown,USA,2024-02-15
3,Raj Patel,India,2024-03-01
4,Emma Wilson,UK,2024-04-01
5,David Lee,Canada,2024-05-10


## Key Points:

- PySpark DataFrames are **immutable**
- You don’t modify → you create a new DataFrame

## Practice:

- Add 2 new customers, from different countries

# 2. READ → View Data

### Purpose:

Retrieve data

In [0]:
#df.show()
df.select("name","country").show()

+-----------+-------+
|       name|country|
+-----------+-------+
| John Smith|     UK|
|Alice Brown|    USA|
|  Raj Patel|  India|
|Emma Wilson|     UK|
|  David Lee| Canada|
+-----------+-------+



In [0]:
# Filtering Data

df.filter(df.country=="Canada").show()

+-----------+---------+-------+-----------+
|customer_id|     name|country|signup_date|
+-----------+---------+-------+-----------+
|          5|David Lee| Canada| 2024-05-10|
+-----------+---------+-------+-----------+



## Key Points:

- `.show() and .display()` displays data
- `.select()` chooses columns
- `.filter()` applies conditions

## Practice:

- Show all customers
- Show only UK customers
- Show only names

# 3. UPDATE → Modify Data

### Purpose:

Update existing data

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
"country",
when(df.customer_id == 1,"USA").otherwise(df.country) #df = df.withColumn("source", lit("manual"))
)
df.display()

customer_id,name,country,signup_date
1,John Smith,USA,2024-01-10
2,Alice Brown,USA,2024-02-15
3,Raj Patel,India,2024-03-01
4,Emma Wilson,UK,2024-04-01
5,David Lee,Canada,2024-05-10


### Important:

- No direct UPDATE like SQL
- You recreate the column

### Key Points:

- Always assign back to `df`
- Logic is applied row-wise

### Practice:

- Update country for one customer
- Update multiple customers

# 4. DELETE → Remove Data

### Purpose:

Delete rows

In [0]:
df = df.filter(df.customer_id!=1)

In [0]:
df=df.filter(df.customer_id>100)

> Can remove almost all data accidentally

### Key Points:

- DELETE = filtering out rows
- No direct DELETE command

### Practice:

- Delete one customer
- Delete customers from UK

# 5. RENAME COLUMNS

Use `withColumnRenamed()` when you want to change the name of an existing column. It returns a new DataFrame.


In [ ]:
df = df.withColumnRenamed("name", "customer_name")
df.show()


### Practice

- Rename `customer_name` back to `name`.


# 6. DROP COLUMNS

Use `drop()` when a column is no longer required. You can drop one or multiple columns.


In [ ]:
df = df.drop("customer_name")
df.show()


### Practice

- Drop the `country` column.


# 7. SORT / ORDER BY

Use `orderBy()` to sort rows. By default, sorting is ascending. Use `desc()` for descending order.


In [ ]:
from pyspark.sql.functions import desc

df.orderBy(desc("customer_id")).show()


### Practice

- Sort customers by `customer_id` in ascending order.


# 8. DISTINCT

Use `distinct()` to remove duplicate rows. You can also select a column first and find its unique values.


In [ ]:
df.select("country").distinct().show()


### Practice

- Show the unique customer countries.


# 9. NULL HANDLING

Use `dropna()` to remove rows containing NULL values and `fillna()` to replace NULL values.


In [ ]:
df = df.fillna("Unknown", subset=["country"])
df.show()


### Practice

- Replace NULL values in `name` with `"Unknown"`.


# 10. GROUP BY AND AGGREGATIONS

Use `groupBy()` with functions such as `count()`, `sum()`, `avg()`, `min()` and `max()` to calculate summaries by group.


In [ ]:
from pyspark.sql.functions import count

df.groupBy("country").agg(
    count("*").alias("customer_count")
).show()


### Practice

- Count how many customers are in each country.


# 11. JOINS

`join()` combines two DataFrames using a matching key. Common types are `inner`, `left`, `right` and `full`.


In [ ]:
orders_data = [
    (101, 1, 250),
    (102, 2, 400),
    (103, 1, 150)
]

orders = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "amount"]
)

result = df.join(orders, "customer_id", "left")
result.show()


### Practice

- Perform an inner join between `df` and `orders` using `customer_id`.


# 12. UNION

Use `union()` or `unionByName()` to combine rows from two DataFrames. `unionByName()` matches columns by name and is generally safer when column order may differ.


In [ ]:
new_data = [(6, "Sarah Khan", "UK", "2024-06-01")]
new_df = spark.createDataFrame(new_data, columns)

df = df.unionByName(new_df)
df.show()


### Practice

- Create a DataFrame with one new customer and combine it with `df` using `unionByName()`.


# 13. CAST DATA TYPES

Use `cast()` to change a column's data type. This is common when reading raw data.


In [ ]:
from pyspark.sql.functions import col

df = df.withColumn("customer_id", col("customer_id").cast("int"))
df = df.withColumn("signup_date", col("signup_date").cast("date"))
df.printSchema()


### Practice

- Cast `customer_id` to `string` and inspect the schema.


# 14. STRING OPERATIONS

PySpark provides functions for common string transformations such as `upper()`, `lower()`, `trim()`, `length()` and `concat_ws()`.


In [ ]:
from pyspark.sql.functions import upper, trim

df = df.withColumn("name", trim("name"))
df = df.withColumn("country", upper("country"))
df.show()


### Practice

- Create a new column containing the customer name in uppercase.


# 15. DATE OPERATIONS

Date functions are commonly used to extract year/month/day and calculate date differences.


In [ ]:
from pyspark.sql.functions import year, month

df = df.withColumn("signup_year", year("signup_date"))
df = df.withColumn("signup_month", month("signup_date"))
df.show()


### Practice

- Create a `signup_year` column from `signup_date`.


# 16. WINDOW FUNCTIONS

Window functions calculate values across related rows without collapsing them like `groupBy()`. They are useful for ranking, latest-record logic, `lag()` and `lead()`.


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("country").orderBy(desc("signup_date"))

df = df.withColumn("rn", row_number().over(window))
df.show()


### Practice

- Create a row number for customers within each country ordered by `signup_date` descending.


# 17. TEMPORARY VIEW

A temporary view lets you query a DataFrame using SQL. It is session-scoped and is not stored as a permanent Unity Catalog object.


In [ ]:
df.createOrReplaceTempView("customers")

spark.sql("""
SELECT *
FROM customers
WHERE country = 'UK'
""").show()


### Practice

- Create a temporary view called `uk_customers` containing only UK customers, then query it with SQL.


# 18. READ DATA

Spark can read common formats such as CSV, Parquet and Delta. In Databricks, Delta tables are especially common.


In [ ]:
# Examples
# csv_df = spark.read.option("header", True).option("inferSchema", True).csv("/path/customers.csv")
# parquet_df = spark.read.parquet("/path/customers")
# delta_df = spark.read.table("catalog.schema.table_name")


### Practice

- Write the PySpark command you would use to read a Delta table named `catalog.schema.customers`.


# 19. WRITE DATA

Use `write` to persist a DataFrame. Common modes are `overwrite`, `append`, `ignore` and `error`.


In [ ]:
# Example - write a DataFrame as a Delta table
# df.write.format("delta").mode("overwrite").saveAsTable("catalog.schema.customers")


### Practice

- Write `df` to a Delta table using append mode.


# 20. REPARTITION AND COALESCE

`repartition()` changes the number of partitions and can redistribute data. `coalesce()` reduces the number of partitions without a full shuffle in typical use. Use these deliberately because they affect performance.


In [ ]:
df_repartitioned = df.repartition(4)
df_repartitioned.rdd.getNumPartitions()

df_coalesced = df_repartitioned.coalesce(2)
df_coalesced.rdd.getNumPartitions()


### Practice

- Repartition `df` into 4 partitions and then reduce it to 2 partitions using `coalesce()`.


# 21. INSPECT A DATAFRAME

These commands are useful when developing and debugging PySpark transformations.


In [ ]:
df.show()
df.printSchema()
print(df.columns)
print(df.dtypes)
print(df.count())
df.describe().show()


### Practice

- Display the schema, columns and row count for `df`.
